In [ ]:
import pandas as pd
import math
import matplotlib.pyplot as plt
import numpy as np
import os
import csv

from PIL import Image, ImageDraw
from matplotlib import cm
from scipy.optimize import curve_fit
from scipy.stats import chisquare
from reconstructionClass import Reconstruction as recon

In [ ]:
## FIMS Fit ##
reconstructionInfo = {
    'Gain': 60,
    'Avalanche Sigma': 20,
    'Pitch': 55,
    'Standoff': 30,
    'Signal Decay Rate': .01,
    'Signal Threshold': 10,
    'File Location': '../Data/Sven_Recoil_Data.root',
    'Tree Name': 'tree_3d_noquant_before_drift'
}
fimsPlot = recon(reconstructionInfo).reconstructFIMS()

In [ ]:
## BEAST Fit ##
#TODO: replace values with proper BEAST values (these are Migdal)
reconstructionInfo = {
    'Gain': 1000,
    'Avalanche Sigma': 300,
    'Pitch': 83,
    'Standoff': 2000,
    'Signal Decay Rate': 5000,
    'Signal Threshold': 200,
    'File Location': '../Data/Sven_Recoil_Data.root',
    'Tree Name': 'tree_3d_noquant_before_drift'
}
beastPlot = recon(reconstructionInfo).reconstructBEAST()

In [ ]:
## Migdal Experiment Fit ##
reconstructionInfo = {
    'Gain': 1000,
    'Avalanche Sigma': 300,
    'Pitch': 83,
    'Standoff': 2000,
    'Signal Decay Rate': 5000,
    'Signal Threshold': 200,
    'File Location': '../Data/Sven_Recoil_Data.root',
    'Tree Name': 'tree_3d_noquant_before_drift'
}
migdalPlot = recon(reconstructionInfo).reconstructMigdal()

In [ ]:
## Testing Area ##
## Information needed for an individual event reconstruction ##
reconstructionInfo = {
    'Gain': 100,
    'Avalanche Sigma': 20,
    'Pitch': 50,
    'Standoff': 50,
    'Signal Decay Rate': 500,
    'Signal Threshold': 20,
    'File Location': '../Data/Sven_Recoil_Data.root',
    'Tree Name': 'tree_3d_noquant_before_drift'
}
reconstruction = recon(reconstructionInfo)

# Extract relevant data from dictionary
pitch = reconstruction.reconInfo['Pitch']
timeRez = 25
transDif = 1600
lonDif = 1000

# Apply Gaussian smear to approximate diffusion
smearData = reconstruction.diffuseData([(0,0,0)], (transDif, transDif, lonDif))

# Discretize data to approximate falling into grid holes
discreteData = reconstruction.discretizeData(smearData, (pitch, pitch, 0))

## Approximate avalanches ##
avalData = discreteData.copy()

# Approximate avalanche
electronID = 0
for xElec,yElec,zElec in discreteData:
    # Multiply each individual electron to approximate gain
    newElectrons = reconstruction.approximateGain((xElec, yElec, zElec))
    
    # Diffuse each new electron to approximate diffusion during avalanche
    if len(newElectrons) > 0:
        #TODO: find better diffusion metric for diffusion incurred during avalanche
        newDiffused = reconstruction.diffuseData(newElectrons, (transDif/100, transDif/100, lonDif/100))
        # Append new electrons to data set
        avalData.extend(newDiffused)

# Discretize data to approximate pixels readout
padData = reconstruction.discretizeData(avalData, (pitch, pitch, timeRez))

# Approximate Signal Readout
readoutData = reconstruction.approximateReadout(padData)